# Leo's voice with VoxCPM2

Generates every spoken phrase of the app in Leo's voice.

- **Engine:** [VoxCPM2](https://github.com/OpenBMB/VoxCPM) by OpenBMB. Code and weights are Apache-2.0 ("free for commercial use").
- **Runtime:** needs an NVIDIA GPU. In Colab: *Runtime → Change runtime type → T4 GPU*.
- **Output:** a zip with mono AAC clips (`.m4a`, which Android, iOS and browsers all play) (slowed for young children, loudness-normalized, silence trimmed),
  the reference clip, and a `manifest.json` describing how everything was made.

**Usual run:** steps 1, 2, 4, 5, 6a, 6b and 7. Leo's voice is already chosen, so step 3 is only for designing a new
voice, and step 6 "Try spellings" is only for fixing a vowel or syllable that sounds wrong.

## 1. Check the GPU

In [ ]:
import sys
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("Python", sys.version.split()[0])

## 2. Install and load VoxCPM2

In [ ]:
!pip install -q voxcpm pyyaml

import json
import os
import random
import shutil
import urllib.request

import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SAMPLE_RATE = model.tts_model.sample_rate
print("Loaded, sample rate", SAMPLE_RATE)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 3. Optional: design a new voice

Skip this step to keep Leo's current voice. Each description is written in parentheses before the text: that is
how VoxCPM2 designs a voice. The first candidate is Leo's current voice, for comparison.

In [ ]:
LEO_VOICE = "Un niño explorador de siete años, cálido y entusiasta, con acento chileno"
LEO_SEED = 2
TEST_PHRASE = "¡Hola! Soy Leo. Ayúdame a encontrar arriba y abajo. Ma. Me. Mi. Mo. Mu."

CANDIDATES = [
    LEO_VOICE,
    "A cheerful six-year-old boy, bright and curious, Latin American Spanish, speaking slowly and clearly",
    "Un niño de seis años, alegre y curioso, con acento chileno, que habla despacio y con claridad",
]
SEEDS = [1, 2, 3]

takes = {}
for number, description in enumerate(CANDIDATES):
    for seed in SEEDS:
        seed_everything(seed)
        takes[(number, seed)] = model.generate(text=f"({description}){TEST_PHRASE}", cfg_value=2.0, inference_timesteps=10)
        print(f"Voice {number}, seed {seed}: {description}")
        display(Audio(takes[(number, seed)], rate=SAMPLE_RATE))

## 4. Leo's voice

The reference clip is what every phrase copies, so Leo always sounds the same. This cell uses, in order:

1. a voice you picked in step 3, if you set `CHOSEN_VOICE` and `CHOSEN_SEED` below;
2. `leo_reference.wav` and `manifest.json` uploaded to the Colab file panel (folder icon on the left);
3. Leo's reference clip from the repository;
4. Leo's voice recreated from its description and seed.

In [ ]:
CHOSEN_VOICE = None  # only if you picked a new voice in step 3, for example 1
CHOSEN_SEED = None  # the seed of that take, for example 2

LEO_VOICE = "Un niño explorador de siete años, cálido y entusiasta, con acento chileno"
LEO_SEED = 2
LEO_REFERENCE_TEXT = "¡Hola! Soy Leo. Ayúdame a encontrar arriba y abajo. Ma. Me. Mi. Mo. Mu."
LEO_REFERENCE_URL = "https://raw.githubusercontent.com/JoseOrtiz/LeeConLeo/main/tool/voice/leo_reference.wav"

os.makedirs("leo_voice", exist_ok=True)
REFERENCE_WAV = "leo_voice/leo_reference.wav"

if CHOSEN_VOICE is not None:
    if CHOSEN_SEED is None:
        raise SystemExit("Set CHOSEN_SEED to the seed of the take you picked in step 3.")
    sf.write(REFERENCE_WAV, takes[(CHOSEN_VOICE, CHOSEN_SEED)], SAMPLE_RATE)
    VOICE_DESCRIPTION, REFERENCE_TEXT = CANDIDATES[CHOSEN_VOICE], TEST_PHRASE
    print("Using the new voice from step 3")
elif os.path.exists("leo_reference.wav") and os.path.exists("manifest.json"):
    earlier = json.load(open("manifest.json", encoding="utf-8"))
    shutil.copy("leo_reference.wav", REFERENCE_WAV)
    VOICE_DESCRIPTION, CHOSEN_SEED, REFERENCE_TEXT = earlier["voiceDescription"], earlier["seed"], earlier["referenceText"]
    print("Using the uploaded voice")
else:
    VOICE_DESCRIPTION, CHOSEN_SEED, REFERENCE_TEXT = LEO_VOICE, LEO_SEED, LEO_REFERENCE_TEXT
    try:
        urllib.request.urlretrieve(LEO_REFERENCE_URL, REFERENCE_WAV)
        print("Using Leo's reference clip from the repository")
    except Exception as error:
        print("Could not download Leo's reference clip:", error)
        seed_everything(LEO_SEED)
        wav = model.generate(text=f"({LEO_VOICE}){LEO_REFERENCE_TEXT}", cfg_value=2.0, inference_timesteps=10)
        sf.write(REFERENCE_WAV, wav, SAMPLE_RATE)
        print("Recreated Leo's voice from its description and seed")

print(VOICE_DESCRIPTION, "| seed", CHOSEN_SEED)
display(Audio(REFERENCE_WAV))

## 5. Load Leo's phrases

Upload `content/prompts.yaml` from your local copy of the project, the branch you are about to open a
pull request from. That way the clips match the phrases in that pull request, including new ones that
are not on `main` yet.

In [ ]:
from google.colab import files
import yaml

print("Choose content/prompts.yaml from your local project")
prompts_yaml = next(iter(files.upload().values())).decode("utf-8")

prompts = yaml.safe_load(prompts_yaml)
SOUNDS = {
    "vowel": ["a", "e", "i", "o", "u"],
    "syllable": ["ma", "me", "mi", "mo", "mu", "pa", "pe", "pi", "po", "pu", "la", "le", "li", "lo", "lu"],
}
phrases = [(prompt_id, index, text) for prompt_id, variants in prompts.items() for index, text in enumerate(variants)]
print(len(phrases), "phrases and", sum(len(texts) for texts in SOUNDS.values()), "vowels and syllables to generate")

## 6a. Set up generation

- Every clip copies the reference voice. No instructions are added to the text: in this mode VoxCPM2 would read them aloud.
- Vowels and syllables are said in series, the way a teacher does it ("Ma... Me... Mi... Mo... Mu."), and cut at the
  pauses, so each one is said deliberately and at full length. If a series doesn't split into the right number of
  sounds, it is tried with periods, and then each sound falls back to a short carrier phrase ("¡Mi!... Escucha bien.").
- `SPELLINGS` says how each sound is written for the engine. A vowel alone is written twice ("Ee"), because a single
  capital letter ("E") is read as an English letter name. Use step 6 "Try spellings" to find a better one.
- Vowels and syllables fade out and end with 0.2 s of silence, so they don't stop abruptly.
- VoxCPM2 has no speed setting, so every clip is slowed with `atempo` (the pitch stays the same). Lower `SPEED` for
  slower speech; 1.0 keeps the original pace.

In [ ]:
import subprocess
import tempfile

from tqdm.auto import tqdm

SPEED = 0.85
SOUND_SECONDS = (0.06, 1.0)
SPELLINGS = {"a": "Aa", "e": "Ee", "i": "Ii", "o": "Oo", "u": "Uu"}


def as_spoken(sound):
    return SPELLINGS.get(sound, sound.capitalize())


def speak(text):
    seed_everything(CHOSEN_SEED)
    return model.generate(
        text=text,
        prompt_wav_path=REFERENCE_WAV,
        prompt_text=REFERENCE_TEXT,
        reference_wav_path=REFERENCE_WAV,
        cfg_value=2.0,
        inference_timesteps=10,
    )


FRAME_SECONDS = 0.01
GAP_SECONDS = 0.18
BLIP_SECONDS = 0.05


def voiced_runs(wav, threshold_db=-40):
    frame = int(SAMPLE_RATE * FRAME_SECONDS)
    count = len(wav) // frame
    rms = np.sqrt(np.mean(wav[: count * frame].reshape(count, frame) ** 2, axis=1)) + 1e-9
    voiced = 20 * np.log10(rms / rms.max()) > threshold_db
    gap = int(GAP_SECONDS / FRAME_SECONDS)
    blip = int(BLIP_SECONDS / FRAME_SECONDS)
    runs = []
    start = None
    silent = 0
    for i, is_voiced in enumerate(voiced):
        if is_voiced:
            if start is None:
                start = i
            silent = 0
        elif start is not None:
            silent += 1
            if silent >= gap:
                runs.append((start, i - silent + 1))
                start = None
                silent = 0
    if start is not None:
        runs.append((start, len(voiced) - silent))
    return frame, [(a * frame, b * frame) for a, b in runs if b - a >= blip]


BEFORE_SECONDS = 0.04
AFTER_SECONDS = 0.08
FADE_SECONDS = 0.06
END_SILENCE_SECONDS = 0.2


def around(wav, start, end):
    return wav[max(0, start - int(BEFORE_SECONDS * SAMPLE_RATE)) : end + int(AFTER_SECONDS * SAMPLE_RATE)]


def first_sound(wav):
    frame, runs = voiced_runs(wav)
    return around(wav, *runs[0]) if len(runs) >= 2 else None


def last_sound(wav):
    frame, runs = voiced_runs(wav)
    return around(wav, *runs[-1]) if len(runs) >= 2 else None


CARRIERS = [("¡{sound}!... Escucha bien.", first_sound), ("Escucha bien... {sound}.", last_sound)]


def fits(sound):
    return SOUND_SECONDS[0] <= len(sound) / SAMPLE_RATE / SPEED <= SOUND_SECONDS[1]


def carried_sound(text):
    for carrier, cut in CARRIERS:
        spoken = carrier.format(sound=as_spoken(text))
        sound = cut(speak(spoken))
        if sound is not None and fits(sound):
            return sound, spoken
    return None, None


SERIES_JOINERS = ["... ", ". "]


def series_groups():
    groups = {"vowels": SOUNDS["vowel"]}
    for syllable in SOUNDS["syllable"]:
        groups.setdefault(syllable[0], []).append(syllable)
    return list(groups.values())


def series_sounds(forms):
    for joiner in SERIES_JOINERS:
        spoken = joiner.join(forms) + "."
        wav = speak(spoken)
        frame, runs = voiced_runs(wav)
        if len(runs) == len(forms):
            return [around(wav, *run) for run in runs], spoken
    return None, None


def audio_filters(soft_ending):
    filters = (
        "silenceremove=start_periods=1:start_threshold=-45dB,areverse,"
        "silenceremove=start_periods=1:start_threshold=-45dB,areverse,"
        f"atempo={SPEED},loudnorm=I=-18:TP=-2"
    )
    if soft_ending:
        filters += f",areverse,afade=t=in:d={FADE_SECONDS},areverse,apad=pad_dur={END_SILENCE_SECONDS}"
    return filters


def encode(wav, clip_path, soft_ending):
    wav_path = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    sf.write(wav_path, wav, SAMPLE_RATE)
    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error", "-i", wav_path, "-af", audio_filters(soft_ending),
            "-ac", "1", "-ar", "48000", "-c:a", "aac", "-b:a", "64k", clip_path,
        ],
        check=True,
    )
    os.remove(wav_path)
    return clip_path


def seconds_of(path):
    probe = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", path]
    return float(subprocess.run(probe, capture_output=True, text=True).stdout.strip() or 0)


print("Ready. Spellings:", SPELLINGS)

## 6. Optional: try spellings

Use this when a vowel or syllable sounds wrong. For each sound it tries several spellings inside the same series
used in 6b, and plays only that sound, processed like the final clips. When one sounds right, write it into
`SPELLINGS` in step 6a, run 6a again, then run 6b. In Spanish, "Y" alone is pronounced like "i".

In [ ]:
TRY = {
    "i": ["Ii", "Y", "Í", "Iii"],
    "pa": ["Pa", "Paa", "Pá", "¡Pa!"],
}

for sound, forms in TRY.items():
    group = next(group for group in series_groups() if sound in group)
    position = group.index(sound)
    print(f"\n{sound}")
    for form in forms:
        cuts, spoken = series_sounds([form if item == sound else as_spoken(item) for item in group])
        if cuts is None:
            print(f"  “{form}”: the series didn't split into {len(group)} parts")
            continue
        preview = encode(cuts[position], tempfile.NamedTemporaryFile(suffix=".m4a", delete=False).name, soft_ending=True)
        print(f"  “{form}” ({seconds_of(preview) - END_SILENCE_SECONDS:.2f} s), from “{spoken}”")
        display(Audio(preview))

## 6b. Generate every clip

In [ ]:
shutil.rmtree("leo_voice/clips", ignore_errors=True)
os.makedirs("leo_voice/clips", exist_ok=True)
manifest = []
not_cut = []
suspicious = []

for prompt_id, index, text in tqdm(phrases):
    file = f"clips/{prompt_id}.{index}.m4a"
    encode(speak(text), f"leo_voice/{file}", soft_ending=False)
    manifest.append({"promptId": prompt_id, "index": index, "text": text, "file": file})

for items in tqdm(series_groups()):
    cuts, series = series_sounds([as_spoken(item) for item in items])
    for position, text in enumerate(items):
        prompt_id = f"{'vowel' if text in SOUNDS['vowel'] else 'syllable'}.{text}"
        if cuts and fits(cuts[position]):
            sound, spoken = cuts[position], series
        else:
            sound, spoken = carried_sound(text)
        if sound is None:
            not_cut.append(prompt_id)
            continue
        file = f"clips/{prompt_id}.0.m4a"
        seconds = seconds_of(encode(sound, f"leo_voice/{file}", soft_ending=True)) - END_SILENCE_SECONDS
        if not SOUND_SECONDS[0] <= seconds <= SOUND_SECONDS[1]:
            suspicious.append(f"{prompt_id} ({seconds:.2f} s)")
        manifest.append({"promptId": prompt_id, "index": 0, "text": text, "file": file, "spokenAs": spoken})

print("Done:", len(manifest), "clips")
if not_cut:
    print("Left out, no clean sound in any series or carrier phrase:", ", ".join(not_cut))
if suspicious:
    print("Check these by ear, their length is unusual for a syllable:", ", ".join(suspicious))

## 7. Listen and download everything

In [ ]:
for prompt_id in ["vowel.e", "vowel.i", "syllable.pa", "syllable.mi", "spatial.up_down.up"]:
    for entry in manifest:
        if entry["promptId"] == prompt_id:
            print(prompt_id, "→", entry["text"])
            display(Audio(f"leo_voice/{entry['file']}"))

with open("leo_voice/manifest.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "engine": "VoxCPM2 (openbmb/VoxCPM2)",
            "license": "Apache-2.0 (code and weights)",
            "voiceDescription": VOICE_DESCRIPTION,
            "seed": CHOSEN_SEED,
            "referenceText": REFERENCE_TEXT,
            "settings": {"cfg_value": 2.0, "inference_timesteps": 10, "speed": SPEED, "spellings": SPELLINGS},
            "clips": manifest,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

shutil.make_archive("leo_voice", "zip", "leo_voice")
from google.colab import files

files.download("leo_voice.zip")